# Fragrantica Dominant Accord Analysis

## Objective

This notebook analyzes the ordering of fragrance accords in the Fragrantica dataset to identify dominant accords and the most common combinations of the first two accords.

Unlike the previous accord analysis, which examined the prevalence of all accords associated with each fragrance, this analysis considers the position of each accord in the original list. The first accord is treated as the primary accord and the second as the secondary accord, subject to validation of the source ordering.

## Analytical Scope

The analysis will cover:

- **Dominant accord distribution:** identify the most frequently occurring first accords across fragrances.
- **Top-two accord combinations:** identify the most common ordered pairs of primary and secondary accords.
- **Gender category comparison:** compare dominant accords and top-two combinations across men's, women's, and unisex fragrances.
- **Data preparation for recommendation systems:** create a structured fragrance profile that can later support personal fragrance wardrobe analysis, preference profiling, and fragrance recommendations.

## Data Source

The analysis uses the cleaned Fragrantica reference dataset stored in Databricks:

`main.fragrance_analytics.fragrantica_perfumes`

The original accord ordering will be preserved. The source data will remain unchanged; any extracted accord positions and analytical results will be created as derived data.

## Methodology and Validation

Before analyzing accord dominance, the notebook will inspect the source schema and sample records to confirm how

In [0]:

%pip install --upgrade "plotly>=6.1.1" "kaleido>=1.0.0"

from pyspark.sql import functions as F

# Load the cleaned Fragrantica reference dataset
source_table = "main.fragrance_analytics.fragrantica_perfumes"
df = spark.table(source_table)

# Exclude fragrances without accords for this analysis
df_with_accords = df.filter(
    F.col("main_accords").isNotNull()
    & (F.trim(F.col("main_accords")) != "[]")
)

# Check the source schema and sample records
df_with_accords.printSchema()
display(df_with_accords.limit(10))

# Validate the filtering results
display(
    df.agg(
        F.count("*").alias("original_fragrances")
    )
    .crossJoin(
        df_with_accords.agg(
            F.count("*").alias("fragrances_with_accords")
        )
    )
    .withColumn(
        "excluded_fragrances",
        F.col("original_fragrances") - F.col("fragrances_with_accords")
    )
)

In [0]:
from pyspark.sql import functions as F

# Parse the accord list into an array of strings
accord_array = F.from_json(
    F.regexp_replace(F.col("main_accords"), "'", '"'),
    "array<string>"
)

# Lowercase and trim each accord while preserving its position
standardized_accords = F.transform(
    accord_array,
    lambda accord: F.lower(F.trim(accord))
)

# Extract the dominant and secondary accords
dominant_accords_df = (
    df_with_accords
    .withColumn("accords", standardized_accords)
    .withColumn("accord_count", F.size("accords"))
    .withColumn("dominant_accord", F.get(F.col("accords"), 0))
    .withColumn("second_accord", F.get(F.col("accords"), 1))
    .withColumn(
        "top_2_profile",
        F.when(
            F.col("accord_count") >= 2,
            F.concat_ws(" → ", "dominant_accord", "second_accord")
        )
    )
)

# Inspect extracted accords and validate the standardization
display(
    dominant_accords_df.select(
        "name",
        "gender",
        "main_accords",
        "accords",
        "accord_count",
        "dominant_accord",
        "second_accord",
        "top_2_profile"
    ).limit(15)
)

In [0]:
from pyspark.sql import functions as F

# Extract distinct accords across all positions
all_accords = (
    dominant_accords_df
    .select(F.explode("accords").alias("accord"))
    .filter(F.col("accord").isNotNull() & (F.trim("accord") != ""))
    .distinct()
)

# Build distinct accord sets for positions 1–5
position_names = ["First", "Second", "Third", "Fourth", "Fifth"]

position_accords = {
    position: (
        dominant_accords_df
        .filter(F.size("accords") >= position)
        .select(F.col("accords")[position - 1].alias("accord"))
        .filter(F.col("accord").isNotNull() & (F.trim("accord") != ""))
        .distinct()
    )
    for position in range(1, 6)
}

# Count distinct accords in each position
summary_rows = [
    ("All distinct accords", all_accords.count())
]

for position in range(1, 6):
    summary_rows.append(
        (
            f"Distinct {position_names[position - 1].lower()} accords",
            position_accords[position].count()
        )
    )

# Count fragrances and distinct ordered dominant–second pairs
two_accords_df = dominant_accords_df.filter(
    F.size("accords") >= 2
)

fragrances_with_two_accords = two_accords_df.count()

distinct_pair_count = (
    two_accords_df
    .select("dominant_accord", "second_accord")
    .distinct()
    .count()
)

summary_rows.extend([
    ("Fragrances with at least two accords", fragrances_with_two_accords),
    ("Distinct dominant–second accord pairs", distinct_pair_count)
])

summary_df = spark.createDataFrame(
    summary_rows,
    ["metric", "unique_count"]
)

display(summary_df)

# Identify accords absent from each position compared with all accords
missing_accords_df = None

for position in range(1, 6):
    missing = (
        all_accords
        .join(position_accords[position], on="accord", how="left_anti")
        .withColumn("position", F.lit(position))
        .withColumn("position_name", F.lit(position_names[position - 1]))
    )

    missing_accords_df = (
        missing if missing_accords_df is None
        else missing_accords_df.unionByName(missing)
    )

display(
    missing_accords_df
    .select("position_name", "accord")
    .orderBy("position", "accord")
)

In [0]:
# Calculate the frequency and percentage of each dominant accord
dominant_accord_distribution = (
    dominant_accords_df
    .filter(F.col("dominant_accord").isNotNull())
    .groupBy("dominant_accord")
    .agg(
        F.count("*").alias("fragrance_count")
    )
    .withColumn(
        "percentage",
        F.round(
            F.col("fragrance_count") /
            dominant_accords_df.count() * 100,
            2
        )
    )
    .orderBy(F.desc("fragrance_count"))
)

# Display the most common dominant accords
display(dominant_accord_distribution.limit(20))

Databricks visualization. Run in Databricks to view.

In [0]:

# Count fragrances with at least two accords
two_accord_fragrance_count = (
    dominant_accords_df
    .filter(F.col("second_accord").isNotNull())
    .count()
)

# Calculate counts and percentages for all ordered accord pairs
accord_pair_distribution = (
    dominant_accords_df
    .filter(F.col("second_accord").isNotNull())
    .groupBy("dominant_accord", "second_accord")
    .agg(
        F.count("*").alias("fragrance_count")
    )
    .withColumn(
        "percentage",
        F.round(
            F.col("fragrance_count") / two_accord_fragrance_count * 100,
            2
        )
    )
    .orderBy(F.desc("fragrance_count"))
)

# Display all accord pairs with their counts and percentages
display(accord_pair_distribution)


In [0]:

import plotly.express as px

# Select the 20 most frequent ordered accord pairs
plot_df = (
    accord_pair_distribution
    .orderBy(F.desc("value") if "value" in accord_pair_distribution.columns
             else F.desc("fragrance_count"))
    .limit(20)
    .toPandas()
)

# Standardize column names for plotting
if "fragrance_count" in plot_df.columns:
    plot_df = plot_df.rename(columns={"fragrance_count": "value"})

plot_df["accord_pair"] = (
    plot_df["dominant_accord"].str.title()
    + " → "
    + plot_df["second_accord"].str.title()
)

# Sort so the most frequent pair appears at the top
plot_df = plot_df.sort_values("value", ascending=True)

# Create an elegant, interactive horizontal bar chart
fig = px.bar(
    plot_df,
    x="value",
    y="accord_pair",
    orientation="h",
    text="value",
    custom_data=["percentage"],
    title="Most Common Dominant–Secondary Accord Pairs",
    labels={
        "value": "Number of fragrances",
        "accord_pair": ""
    },
    color_discrete_sequence=["#9A6A78"]
)

fig.update_traces(
    texttemplate="%{text:,}",
    textposition="outside",
    hovertemplate=(
        "<b>%{y}</b><br>"
        "Fragrances: %{x:,}<br>"
        "Share of eligible fragrances: %{customdata[0]:.2f}%"
        "<extra></extra>"
    )
)

fig.update_layout(
    template="plotly_white",
    paper_bgcolor="#F7F3EF",
    plot_bgcolor="#F7F3EF",
    font=dict(family="Arial", color="#2F2930", size=12),
    title=dict(
        font=dict(size=21, color="#2F2930"),
        x=0.02,
        xanchor="left"
    ),
    xaxis=dict(
        showgrid=True,
        gridcolor="#DDD4D7",
        zeroline=False
    ),
    yaxis=dict(
        showgrid=False,
        categoryorder="array",
        categoryarray=plot_df["accord_pair"].tolist()
    ),
    margin=dict(l=170, r=70, t=90, b=60),
    height=720,
    showlegend=False
)

displayHTML(
    fig.to_html(
        full_html=False,
        include_plotlyjs=True
    )
)


In [0]:

import plotly.express as px

# Prepare the top 20 complete accord pairs
plot_df = (
    accord_pair_distribution
    .filter(
        F.col("dominant_accord").isNotNull()
        & F.col("second_accord").isNotNull()
    )
    .orderBy(F.desc("fragrance_count"))
    .limit(20)
    .toPandas()
)

plot_df["dominant_label"] = plot_df["dominant_accord"].str.title()
plot_df["secondary_label"] = plot_df["second_accord"].str.title()
plot_df["pair_label"] = (
    plot_df["dominant_label"] + " → " + plot_df["secondary_label"]
)

# Create a treemap grouped by dominant and secondary accord
fig = px.treemap(
    plot_df,
    path=["dominant_label", "secondary_label"],
    values="fragrance_count",
    color="fragrance_count",
    color_continuous_scale=[
        "#F0E5E5", "#D4B5BF", "#B99AA5",
        "#9A6A78", "#674653", "#2F2930"
    ],
    custom_data=["pair_label", "percentage", "fragrance_count"]
)

fig.update_traces(
    texttemplate="%{label}<br>%{value:,}",
    marker=dict(line=dict(color="#F7F3EF", width=3)),
    hovertemplate=(
        "<b>%{customdata[0]}</b><br>"
        "Fragrances: %{customdata[2]:,}<br>"
        "Share: %{customdata[1]:.2f}%"
        "<extra></extra>"
    ),
    tiling=dict(packing="squarify", pad=3)
)

fig.update_layout(
    title=dict(
        text="Dominant–Secondary Accord Profiles",
        x=0.03,
        font=dict(size=23, color="#2F2930")
    ),
    paper_bgcolor="#F7F3EF",
    font=dict(family="Arial", color="#2F2930"),
    margin=dict(t=85, l=15, r=15, b=20),
    height=720,
    coloraxis_showscale=False
)

displayHTML(
    fig.to_html(
        full_html=False,
        include_plotlyjs=True
    )
)


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Parse and lowercase accord labels while preserving their order
accord_array = F.from_json(
    F.regexp_replace(F.col("main_accords"), "'", '"'),
    "array<string>"
)

accords_df = (
    df_with_accords
    .withColumn(
        "accords",
        F.transform(accord_array, lambda x: F.lower(F.trim(x)))
    )
    .withColumn("dominant_accord", F.get("accords", 0))
    .withColumn("second_accord", F.get("accords", 1))
    .withColumn(
        "third_accord",
        F.coalesce(F.get("accords", 2), F.lit("(no third accord)"))
    )
    .filter(F.col("dominant_accord").isNotNull())
)

# Count fragrances for each three-level accord combination
accord_hierarchy = (
    accords_df
    .groupBy("dominant_accord", "second_accord", "third_accord")
    .agg(F.count("*").alias("fragrance_count"))
)

# Calculate dominant-level and dominant-second-level totals
accord_hierarchy = (
    accord_hierarchy
    .withColumn(
        "dominant_total",
        F.sum("fragrance_count").over(
            Window.partitionBy("dominant_accord")
        )
    )
    .withColumn(
        "pair_total",
        F.sum("fragrance_count").over(
            Window.partitionBy("dominant_accord", "second_accord")
        )
    )
    .withColumn(
        "second_pct_of_dominant",
        F.round(
            F.col("pair_total") / F.col("dominant_total") * 100, 2
        )
    )
    .withColumn(
        "third_pct_of_pair",
        F.round(
            F.col("fragrance_count") / F.col("pair_total") * 100, 2
        )
    )
)

# Validate the hierarchy before plotting
display(
    accord_hierarchy
    .orderBy(F.desc("dominant_total"), F.desc("fragrance_count"))
    .limit(20)
)

In [0]:

import pandas as pd
import plotly.graph_objects as go
from pyspark.sql import functions as F

# --------------------------------------------------
# Fragrance accord hierarchy visualization
# --------------------------------------------------

BG = "#F7F3EF"
BORDEAUX = "#674653"
ROSE = "#D4B5BF"
MAUVE = "#9A6A78"
LIGHT_ROSE = "#E7D3D8"
WHITE = "#FFFFFF"


def plot_accord_hierarchy(dominant_accord):
    """
    Three-level hierarchy:
    dominant accord -> second accord -> third accord.

    Second-level percentages: share of the dominant group.
    Third-level percentages: share within the second-level group.
    Displays the top 5 third-level accords per branch.
    """

    # 1. Retrieve the selected dominant accord
    pdf = (
        accord_hierarchy
        .filter(F.col("dominant_accord") == dominant_accord)
        .select(
            "second_accord",
            "third_accord",
            "fragrance_count"
        )
        .toPandas()
    )

    if pdf.empty:
        print(f"No data found for dominant accord: {dominant_accord}")
        return

    pdf["second_accord"] = (
        pdf["second_accord"]
        .fillna("(no second accord)")
        .astype(str)
    )
    pdf["third_accord"] = (
        pdf["third_accord"]
        .fillna("(no third accord)")
        .astype(str)
    )
    pdf["fragrance_count"] = pdf["fragrance_count"].astype(int)

    # 2. Calculate totals and hierarchy
    root_total = int(pdf["fragrance_count"].sum())

    second_counts = (
        pdf.groupby("second_accord")["fragrance_count"]
        .sum()
        .sort_values(ascending=False)
    )

    second_top = second_counts.head(5)
    second_other = int(second_counts.iloc[5:].sum())

    second_items = []

    for second, pair_total in second_top.items():
        pair_total = int(pair_total)

        third_counts = (
            pdf.loc[pdf["second_accord"] == second]
            .groupby("third_accord")["fragrance_count"]
            .sum()
            .sort_values(ascending=False)
        )

        # Top 5 third-level accords
        third_top = third_counts.head(5)
        third_other = int(third_counts.iloc[5:].sum())

        third_items = [
            {
                "name": str(name),
                "count": int(count),
                "pct": int(count) / pair_total * 100
            }
            for name, count in third_top.items()
        ]

        if third_other > 0:
            third_items.append({
                "name": "Other",
                "count": third_other,
                "pct": third_other / pair_total * 100
            })

        second_items.append({
            "name": str(second),
            "count": pair_total,
            "pct": pair_total / root_total * 100,
            "third": third_items
        })

    if second_other > 0:
        second_items.append({
            "name": "Other",
            "count": second_other,
            "pct": second_other / root_total * 100,
            "third": []
        })

    # 3. Prepare figure
    fig = go.Figure()
    shapes = []
    annotations = []

    def add_card(x0, y0, x1, y1, fill, line=None):
        shapes.append({
            "type": "rect",
            "xref": "paper",
            "yref": "paper",
            "x0": x0, "y0": y0,
            "x1": x1, "y1": y1,
            "fillcolor": fill,
            "line": {"color": line or fill, "width": 1},
            "layer": "below"
        })

    def add_text(
        x, y, text, size=13, color=BORDEAUX,
        anchor="center", align="center"
    ):
        annotations.append({
            "xref": "paper",
            "yref": "paper",
            "x": x, "y": y,
            "text": text,
            "showarrow": False,
            "xanchor": anchor,
            "yanchor": "middle",
            "align": align,
            "font": {
                "family": "Arial",
                "size": size,
                "color": color
            }
        })

    # 4. Root card: narrower
    add_card(0.38, 0.88, 0.62, 0.975, BORDEAUX)

    add_text(
        0.5, 0.944,
        f"<b>{dominant_accord.title()}</b>",
        size=23, color=WHITE
    )
    add_text(
        0.5, 0.907,
        f"{root_total:,} fragrances  ·  100%",
        size=14, color=WHITE
    )

    # 5. Position second-level branches
    n = len(second_items)

    centers = (
        [0.5] if n == 1
        else [0.085 + i * 0.83 / (n - 1) for i in range(n)]
    )

    # Narrower cards
    col_width = min(0.115, 0.82 / max(n, 1))

    # Root connector and horizontal rail
    shapes.append({
        "type": "line",
        "xref": "paper", "yref": "paper",
        "x0": 0.5, "y0": 0.88,
        "x1": 0.5, "y1": 0.85,
        "line": {"color": MAUVE, "width": 2}
    })

    if n > 1:
        shapes.append({
            "type": "line",
            "xref": "paper", "yref": "paper",
            "x0": min(centers), "y0": 0.85,
            "x1": max(centers), "y1": 0.85,
            "line": {"color": MAUVE, "width": 2}
        })

    # 6. Draw second-level cards and third-level rows
    for i, item in enumerate(second_items):
        x = centers[i]
        x0 = x - col_width / 2
        x1 = x + col_width / 2

        # Connector to second-level card
        shapes.append({
            "type": "line",
            "xref": "paper", "yref": "paper",
            "x0": x, "y0": 0.85,
            "x1": x, "y1": 0.80,
            "line": {"color": MAUVE, "width": 1.5}
        })

        # Second-level card
        add_card(x0, 0.715, x1, 0.80, LIGHT_ROSE, ROSE)

        second_label = item["name"]
        if len(second_label) > 14:
            second_label = second_label[:12] + "…"

        # Category name
        add_text(
            x, 0.773,
            f"<b>{second_label}</b>",
            size=15
        )

        # Metrics on separate lines
        add_text(
            x, 0.737,
            f"{item['pct']:.1f}% of root",
            size=12
        )
        add_text(
            x, 0.721,
            f"n = {item['count']:,}",
            size=11
        )

        # 7. Third-level cards
        third = item["third"]

        if not third:
            add_text(
                x, 0.65,
                "Grouped remainder",
                size=11, color=MAUVE
            )
            continue

        # Connector from parent to third-level rows
        shapes.append({
            "type": "line",
            "xref": "paper", "yref": "paper",
            "x0": x, "y0": 0.715,
            "x1": x, "y1": 0.685,
            "line": {"color": ROSE, "width": 1.5}
        })

        row_top = 0.645
        row_gap = 0.105
        inner_pad = col_width * 0.025

        for j, child in enumerate(third):
            cy = row_top - j * row_gap

            rx0 = x0 + inner_pad
            rx1 = x1 - inner_pad

            # Third-level card
            add_card(
                rx0, cy - 0.046,
                rx1, cy + 0.046,
                WHITE, LIGHT_ROSE
            )

            label = child["name"]
            if len(label) > 13:
                label = label[:11] + "…"

            # Category name
            add_text(
                x, cy + 0.026,
                f"<b>{label}</b>",
                size=13
            )

            # Percentage and count on separate lines
            add_text(
                x, cy + 0.002,
                f"{child['pct']:.1f}% of parent",
                size=11
            )
            add_text(
                x, cy - 0.019,
                f"n = {child['count']:,}",
                size=10
            )

            # Proportional bar
            bar_x0 = rx0 + 0.006
            bar_x1 = rx1 - 0.006
            bar_y0 = cy - 0.037
            bar_y1 = cy - 0.032

            shapes.append({
                "type": "rect",
                "xref": "paper", "yref": "paper",
                "x0": bar_x0, "y0": bar_y0,
                "x1": bar_x1, "y1": bar_y1,
                "fillcolor": LIGHT_ROSE,
                "line": {"width": 0},
                "layer": "below"
            })

            bar_fraction = min(max(child["pct"] / 100, 0), 1)

            if bar_fraction > 0:
                shapes.append({
                    "type": "rect",
                    "xref": "paper", "yref": "paper",
                    "x0": bar_x0,
                    "y0": bar_y0,
                    "x1": bar_x0 + (bar_x1 - bar_x0) * bar_fraction,
                    "y1": bar_y1,
                    "fillcolor": ROSE if j % 2 == 0 else MAUVE,
                    "line": {"width": 0},
                    "layer": "below"
                })

    # 8. Final layout
    fig.update_layout(
        title={
            "text": (
                f"<b>{dominant_accord.title()} Accord Hierarchy</b>"
                f"<br><sup>Second-level share of dominant accord · "
                f"Third-level share of its parent</sup>"
            ),
            "x": 0.5,
            "xanchor": "center",
            "font": {"color": BORDEAUX, "size": 24}
        },
        paper_bgcolor=BG,
        plot_bgcolor=BG,
        shapes=shapes,
        annotations=annotations,
        width=1450,
        height=1050,
        margin={"l": 25, "r": 25, "t": 115, "b": 25},
        xaxis={"visible": False, "range": [0, 1]},
        yaxis={"visible": False, "range": [0, 1]},
        showlegend=False
    )

    # Display visualization
    displayHTML(
        fig.to_html(
            full_html=False,
            include_plotlyjs="cdn",
            config={
                "displayModeBar": False,
                "responsive": True
            }
        )
    )

    # Save standalone interactive HTML
    # (Kaleido v1 needs Chrome, unavailable on this arm64 platform; HTML is fully portable)
    html_path = f"/tmp/{dominant_accord.replace(' ', '_')}_hierarchy.html"

    with open(html_path, "w") as f:
        f.write(fig.to_html(
            full_html=True,
            include_plotlyjs=True,
            config={"displayModeBar": False, "responsive": True}
        ))

    print(f"HTML saved to: {html_path}")

    return html_path

In [0]:
plot_accord_hierarchy("woody")
plot_accord_hierarchy("citrus")
plot_accord_hierarchy("fruity")
plot_accord_hierarchy("white floral")
plot_accord_hierarchy("floral")
plot_accord_hierarchy("aromatic")
plot_accord_hierarchy("amber")
plot_accord_hierarchy("warm spicy")
plot_accord_hierarchy("sweet")
# plot_accord_hierarchy("vanilla")
plot_accord_hierarchy("rose")
plot_accord_hierarchy("powdery")
plot_accord_hierarchy("musky")
plot_accord_hierarchy("fresh spicy")
plot_accord_hierarchy("green")
plot_accord_hierarchy("oud")
plot_accord_hierarchy("leather")
plot_accord_hierarchy("lavender")
plot_accord_hierarchy("patchouli")
plot_accord_hierarchy("aquatic")

In [0]:
import os
import glob
import shutil

source_dir = "/tmp"
destination_dir = (
    "/Volumes/main/fragrance_analytics/reference_data/visualizations/"
)

os.makedirs(destination_dir, exist_ok=True)

html_files = glob.glob(
    os.path.join(source_dir, "*hierarchy*.html")
)

if not html_files:
    print("No matching hierarchy HTML files found in /tmp.")
else:
    total_size = 0

    for source in sorted(html_files):
        filename = os.path.basename(source)
        destination = os.path.join(destination_dir, filename)

        shutil.copy2(source, destination)

        size = os.path.getsize(destination)
        total_size += size

        print(f"Copied: {filename} ({size / (1024 * 1024):.2f} MB)")

    print(f"\nFiles copied: {len(html_files)}")
    print(f"Total size: {total_size / (1024 * 1024):.2f} MB")
    print(f"Destination: {destination_dir}")